# FaultSense - CWRU LoRA + RAG Pipeline V2

This notebook continues from `FaultSense_CWRU_EndToEnd_V1.ipynb`.

It assumes fine-tuning is already complete and uses the saved adapter at:

`/content/drive/MyDrive/FaultSense_Project/finetune/r8_all7_2ep`

The RAG corpus is expected at:

`/content/drive/MyDrive/FaultSense_Project/rag/corpus`


---
## 1. Mount Drive and Configure Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/FaultSense_Project')
RAG_DIR = PROJECT_DIR / 'rag'
RAG_CORPUS_DIR = RAG_DIR / 'corpus'
RAG_INDEX_DIR = RAG_DIR / 'index'
FINETUNE_DIR = PROJECT_DIR / 'finetune'
ADAPTER_DIR = FINETUNE_DIR / 'r8_all7_2ep'
TRAIN_JSON = PROJECT_DIR / 'train_cwru.json'
TEST_JSON = PROJECT_DIR / 'test_cwru.json'

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

for path in [PROJECT_DIR, RAG_CORPUS_DIR, FINETUNE_DIR, ADAPTER_DIR, TRAIN_JSON, TEST_JSON]:
    print(f'{path}:', 'OK' if path.exists() else 'MISSING')

RAG_INDEX_DIR.mkdir(parents=True, exist_ok=True)
print('RAG index directory:', RAG_INDEX_DIR)

/content/drive/MyDrive/FaultSense_Project: OK
/content/drive/MyDrive/FaultSense_Project/rag/corpus: MISSING
/content/drive/MyDrive/FaultSense_Project/finetune: MISSING
/content/drive/MyDrive/FaultSense_Project/finetune/r8_all7_2ep: MISSING
/content/drive/MyDrive/FaultSense_Project/train_cwru.json: MISSING
/content/drive/MyDrive/FaultSense_Project/test_cwru.json: MISSING
RAG index directory: /content/drive/MyDrive/FaultSense_Project/rag/index


---
## 2. Install Dependencies

The LoRA classifier uses `transformers`, `peft`, `bitsandbytes`, and `accelerate`; the RAG index uses `PyMuPDF`, `sentence-transformers`, and `faiss-cpu`.

In [4]:
!pip install -q "transformers>=4.44,<4.50" "peft>=0.11,<0.13" "accelerate>=0.33" \
  "bitsandbytes>=0.43" "sentence-transformers>=2.7" "faiss-cpu>=1.8" \
  "PyMuPDF>=1.24" pandas numpy scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 126.5 MB/s eta 0:00:00


---
## 3. Inspect Existing Artifacts

In [7]:
import json
from pprint import pprint

def load_json_records(path):
    text = Path(path).read_text(encoding='utf-8').strip()
    if not text:
        return []
    if text[0] == '[':
        return json.loads(text)
    return [json.loads(line) for line in text.splitlines() if line.strip()]

test_records = load_json_records(TEST_JSON)
print('Test records:', len(test_records))
print('\nSample test record:')
pprint(test_records[0])

print('\nPDF corpus:')
for pdf in sorted(RAG_CORPUS_DIR.glob('*.pdf')):
    print('-', pdf.name)

print('\nFinetune folders:')
for item in sorted(FINETUNE_DIR.iterdir()):
    if item.is_dir():
        print('-', item.name)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FaultSense_Project/test_cwru.json'

---
## 4. Build or Load FAISS RAG Index

This section turns the maintenance PDFs into a searchable vector index.
Each PDF is parsed page by page, split using fixed-size chunking with overlap (`chunk_size=220` words and `overlap=40` words), embedded with MiniLM, and stored in FAISS.
Retrieval is semantic retrieval: the query and chunks are embedded in the same vector space, then FAISS returns the most similar chunks.

If the index already exists in `rag/index`, the notebook loads it instead of rebuilding it.

In [8]:
import re
import numpy as np
import pandas as pd
import fitz
import faiss
from sentence_transformers import SentenceTransformer

CHUNKS_PATH = RAG_INDEX_DIR / 'chunks.jsonl'
FAISS_PATH = RAG_INDEX_DIR / 'manuals.faiss'
RAG_CONFIG_PATH = RAG_INDEX_DIR / 'config.json'

def clean_text(text):
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def extract_pdf_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        text = clean_text(page.get_text('text'))
        if text:
            pages.append({'source': pdf_path.name, 'page': page.number + 1, 'text': text})
    return pages

def chunk_words(text, chunk_size=220, overlap=40):
    words = text.split()
    step = max(1, chunk_size - overlap)
    for start in range(0, len(words), step):
        window = words[start:start + chunk_size]
        if len(window) >= 40:
            yield start, ' '.join(window)

def build_chunks(corpus_dir):
    chunks = []
    for pdf_path in sorted(corpus_dir.glob('*.pdf')):
        for page in extract_pdf_pages(pdf_path):
            for start, text in chunk_words(page['text']):
                chunks.append({
                    'id': f"{pdf_path.stem}:p{page['page']}:w{start}",
                    'source': page['source'],
                    'page': page['page'],
                    'text': text,
                })
    return chunks

def save_chunks(chunks, path):
    with open(path, 'w', encoding='utf-8') as f:
        for chunk in chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

def load_chunks(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def build_or_load_rag_index(force_rebuild=False):
    encoder = SentenceTransformer(EMBEDDING_MODEL)
    if FAISS_PATH.exists() and CHUNKS_PATH.exists() and not force_rebuild:
        print('Loading existing FAISS index...')
        chunks = load_chunks(CHUNKS_PATH)
        index = faiss.read_index(str(FAISS_PATH))
        return encoder, index, chunks

    print('Building chunks from PDFs...')
    chunks = build_chunks(RAG_CORPUS_DIR)
    print('Chunks:', len(chunks))
    save_chunks(chunks, CHUNKS_PATH)

    print('Embedding chunks...')
    embeddings = encoder.encode(
        [c['text'] for c in chunks],
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    embeddings = np.asarray(embeddings, dtype='float32')
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, str(FAISS_PATH))
    RAG_CONFIG_PATH.write_text(json.dumps({'embedding_model': EMBEDDING_MODEL}, indent=2), encoding='utf-8')
    print('Saved FAISS index:', FAISS_PATH)
    return encoder, index, chunks

rag_encoder, rag_index, rag_chunks = build_or_load_rag_index(force_rebuild=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building chunks from PDFs...
Chunks: 0
Embedding chunks...


Batches: 0it [00:00, ?it/s]

IndexError: tuple index out of range

---
## 5. Retrieval-Only Smoke Test

This tests the RAG system before involving the fine-tuned model.
The helper embeds a maintenance query, retrieves the most similar manual chunks, and prints source PDF, page number, score, and text preview.
Good results here mean the corpus and FAISS index are working.

In [ ]:
def retrieve_manual_context(query, top_k=4):
    query_vec = rag_encoder.encode([query], normalize_embeddings=True)
    query_vec = np.asarray(query_vec, dtype='float32')
    scores, indices = rag_index.search(query_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        chunk = rag_chunks[int(idx)]
        results.append({
            'score': float(score),
            'source': chunk['source'],
            'page': chunk['page'],
            'text': chunk['text'],
        })
    return results

for query in [
    'inner race bearing fault repair',
    'outer race bearing vibration maintenance',
    'ball bearing defect lubrication inspection',
]:
    print('\nQUERY:', query)
    for result in retrieve_manual_context(query, top_k=3):
        print(f"score={result['score']:.3f} | {result['source']} | page {result['page']}")
        print(result['text'][:500].replace('\n', ' '), '...')


QUERY: inner race bearing fault repair
score=0.594 | 14219_3_EN_-_Bearing_failures_LOW_pdf_preview_medium.pdf | page 5
attain its expected service life . Failures will generally cause eco- nomic losses due to loss of production, consequential damage of adjacent parts, and the cost of repairs . Unexpected bearing failure can occur for a variety of reasons . Each failure leaves its own special imprint on the bearing . By examining a failed or damaged bear- ing, it is possible in the majority of cases to establish the root cause and define cor- rective actions to prevent a recurrence . This publication is intended  ...
score=0.562 | wl_82102_3_de_en.pdf | page 11
directly behind filters (wrong concentration of particles) Should a bearing be removed from a machine due to damage the cause of the latter must be clarified as well as the me- ans to avoid future failure. For the most reliable results possible it is practical to follow a systematic procedure when se- curing and inspecting the b

---
## 6. Load Fine-Tuned LoRA Classifier

This loads the base Qwen2.5 instruction model in 4-bit mode and attaches the saved LoRA adapter from `finetune/r8_all7_2ep`.
The loaded model is used only for CWRU fault classification; repair context still comes from the RAG retriever.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()
print('Loaded adapter:', ADAPTER_DIR)
print('Device:', model.device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded adapter: /content/drive/MyDrive/FaultSense_Project/finetune/r8_all7_2ep
Device: cuda:0


---
## 7. LoRA Fault Prediction

These helpers reproduce the exact Alpaca-style prompt used during fine-tuning.
The model output is normalized so labels with en dashes and ASCII hyphens compare correctly.
The final lines run one held-out test example to verify the adapter can predict a fault label.

In [ ]:
VALID_LABELS = [
    'Ball - Low', 'Ball - Medium', 'Ball - High',
    'Inner Race - Low', 'Inner Race - Medium', 'Inner Race - High',
    'Outer Race - Low', 'Outer Race - Medium', 'Outer Race - High',
    'Normal - Healthy',
]

def normalize_label(text):
    return ' '.join(str(text).replace('–', '-').replace('—', '-').split())

def extract_label(generated_text):
    text = normalize_label(generated_text)
    for label in VALID_LABELS:
        if text.startswith(label):
            return label
    for label in VALID_LABELS:
        if label in text:
            return label
    return text.split('\n')[0].strip()

def build_fault_prompt(instruction, sensor_input):
    return (
        '### Instruction:\n'
        f'{instruction}\n\n'
        '### Input:\n'
        f'{sensor_input}\n\n'
        '### Response:\n'
    )

def predict_fault_label(instruction, sensor_input, max_new_tokens=24):
    prompt = build_fault_prompt(instruction, sensor_input)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return extract_label(generated)

sample = test_records[0]
pred = predict_fault_label(sample['instruction'], sample['input'])
print('Input:', sample['input'])
print('Expected:', normalize_label(sample['output']))
print('Predicted:', pred)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Input: Max=3.9313, Min=-4.1508, Mean=0.0148, StdDev=0.6629, RMS=0.6629, Skewness=-0.2887, Kurtosis=8.6678, CrestFactor=5.9304, FormFactor=44.7884
Expected: Outer Race - High
Predicted: Outer Race - High


---
## 8. LoRA-Only Evaluation

This evaluates classification accuracy on held-out CWRU examples without RAG.
It computes exact-match accuracy and per-class accuracy, which is the primary quantitative metric for the fine-tuned classifier.
The default limit is 50 for a quick run; set `limit=None` for the full test set.

In [ ]:
from collections import defaultdict

def evaluate_lora(records, limit=None):
    if limit is not None:
        records = records[:limit]
    correct = 0
    per_class = defaultdict(lambda: {'correct': 0, 'total': 0})
    rows = []
    for i, ex in enumerate(records):
        expected = normalize_label(ex['output'])
        predicted = normalize_label(predict_fault_label(ex['instruction'], ex['input']))
        ok = predicted == expected
        correct += int(ok)
        per_class[expected]['total'] += 1
        per_class[expected]['correct'] += int(ok)
        rows.append({'index': i, 'expected': expected, 'predicted': predicted, 'correct': ok})
        if (i + 1) % 25 == 0:
            print(f'Evaluated {i + 1}/{len(records)}')
    total = len(records)
    print(f'Overall accuracy: {correct}/{total} = {correct / total:.2%}')
    print('\nPer-class accuracy:')
    for label in sorted(per_class):
        c = per_class[label]['correct']
        t = per_class[label]['total']
        print(f'{label:<24} {c:>3}/{t:<3} {c / t:>7.2%}')
    return rows

# Start with a small limit. Increase to None for the full held-out test set.
eval_rows = evaluate_lora(test_records, limit=50)

Evaluated 25/50
Evaluated 50/50
Overall accuracy: 45/50 = 90.00%

Per-class accuracy:
Ball - High                6/8    75.00%
Ball - Low                 3/3   100.00%
Ball - Medium              3/4    75.00%
Inner Race - High          3/3   100.00%
Inner Race - Low           5/5   100.00%
Inner Race - Medium        6/6   100.00%
Normal - Healthy           5/5   100.00%
Outer Race - High          7/8    87.50%
Outer Race - Low           2/2   100.00%
Outer Race - Medium        5/6    83.33%



## 9. Chained FaultSense Pipeline

This is the full two-stage FaultSense pipeline.

**Stage 1** uses the fine-tuned LoRA adapter to classify the CWRU sensor features into a fault label such as `Outer Race - High`.

**Stage 2** uses that label to build a severity-aware expanded query, retrieve relevant manual passages from the FAISS RAG index, then asks the base instruction model to generate repair advice from only those retrieved passages.

The generated response cites the retrieved chunks so the advice stays grounded in the manual context.

The adapter is intentionally disabled during the repair-advice generation step when PEFT supports `model.disable_adapter()`. This keeps the LoRA adapter focused on classification while the base model handles longer natural-language RAG responses with citations.

**Note**: Can consider deduplication of words in the RAG query after query expansion.

In [ ]:
def parse_fault_label(label):
    label = normalize_label(label)
    if ' - ' in label:
        fault_type, severity = label.split(' - ', 1)
    else:
        fault_type, severity = label, 'Unknown'
    return fault_type, severity

def label_to_repair_query(label):
    fault_type, severity = parse_fault_label(label)
    expansions = {
        'Ball': 'bearing ball defect vibration lubrication inspection replacement',
        'Inner Race': 'bearing inner race defect vibration spalling repair replacement',
        'Outer Race': 'bearing outer race defect vibration housing alignment repair replacement',
        'Normal': 'normal bearing condition preventive maintenance lubrication inspection',
    }
    fault_keywords = expansions.get(fault_type, fault_type)
    severity_keywords = {
        'Low': 'monitor inspection preventive maintenance routine check',
        'Medium': 'maintenance repair inspect soon planned replacement',
        'High': 'immediate shutdown critical urgent replacement safety risk',
    }.get(severity, '')
    return f"{fault_type} bearing fault {severity} severity {fault_keywords} {severity_keywords}".strip()

def format_context_for_generation(contexts, max_chars_per_chunk=1200):
    blocks = []
    for i, ctx in enumerate(contexts, start=1):
        text = ctx['text'][:max_chars_per_chunk].replace('\n', ' ')
        blocks.append(
            f"[Source {i}] {ctx['source']} page {ctx['page']} score={ctx['score']:.3f}\n{text}"
        )
    return '\n\n'.join(blocks)

def citation_list(contexts, limit=3):
    return [
        {
            'source_id': i + 1,
            'source': c['source'],
            'page': c['page'],
            'score': c['score'],
        }
        for i, c in enumerate(contexts[:limit])
    ]

def build_repair_prompt(label, sensor_input, contexts):
    fault_type, severity = parse_fault_label(label)
    context_block = format_context_for_generation(contexts)
    return f"""You are FaultSense, an industrial bearing maintenance assistant.

Use only the retrieved maintenance-manual context below to write repair guidance.
Do not invent procedures that are not supported by the context. If the context is incomplete, say what should be inspected or verified rather than making unsupported claims.

Predicted diagnosis: {fault_type} bearing fault
Severity: {severity}
Sensor features: {sensor_input}

Retrieved maintenance-manual context:
{context_block}

Write a concise recommendation with these fields:
1. Likely issue
2. Immediate action
3. Inspection or repair steps
4. Operating priority
5. Citations by source number

Set operating priority according to the predicted severity:
- Low severity: low priority; monitor closely and schedule inspection.
- Medium severity: medium priority; schedule maintenance soon.
- High severity: high priority; take immediate action or reduce operation.
Do not upgrade or downgrade the priority unless the retrieved context explicitly supports it.

Cite at least two source numbers when the retrieved context supports the answer.
If only one source is actually used, briefly say why.
"""

def generate_repair_advice(label, sensor_input, contexts, max_new_tokens=220):
    prompt = build_repair_prompt(label, sensor_input, contexts)
    messages = [
        {
            'role': 'system',
            'content': 'You answer using only the supplied maintenance-manual context and cite source numbers.',
        },
        {'role': 'user', 'content': prompt},
    ]
    if hasattr(tokenizer, 'apply_chat_template'):
        text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text_prompt = prompt + '\nRecommendation:\n'

    inputs = tokenizer(text_prompt, return_tensors='pt', truncation=True, max_length=3500).to(model.device)

    # The adapter was trained for fault labels. For advice generation, prefer the base model if PEFT supports temporarily disabling the adapter.
    try:
        generation_context = model.disable_adapter()
    except Exception:
        generation_context = None

    with torch.no_grad():
        if generation_context is None:
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        else:
            with generation_context:
                output = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def make_recommendation(label, sensor_input, contexts):
    fault_type, severity = parse_fault_label(label)
    generated_advice = generate_repair_advice(label, sensor_input, contexts)
    return {
        'diagnosis': f'{fault_type} bearing fault at {severity} severity',
        'rag_generated_advice': generated_advice,
        'manual_citations': citation_list(contexts),
    }

def faultsense_diagnose(instruction, sensor_input, top_k=4):
    label = predict_fault_label(instruction, sensor_input)
    query = label_to_repair_query(label)
    contexts = retrieve_manual_context(query, top_k=top_k)
    recommendation = make_recommendation(label, sensor_input, contexts)
    return {
        'fault_label': label,
        'rag_query': query,
        'recommendation': recommendation,
        'retrieved_context': contexts,
    }

demo = faultsense_diagnose(test_records[0]['instruction'], test_records[0]['input'])
pprint(demo['recommendation'])
print('\nTop retrieved context:')
for ctx in demo['retrieved_context'][:2]:
    print(f"- {ctx['source']} page {ctx['page']} score={ctx['score']:.3f}")
    print(ctx['text'][:500], '\n')



{'diagnosis': 'Outer Race bearing fault at High severity',
 'manual_citations': [{'page': 97,
                       'score': 0.5874398946762085,
                       'source': '14219_3_EN_-_Bearing_failures_LOW_pdf_preview_medium.pdf',
                       'source_id': 1},
                      {'page': 19,
                       'score': 0.5756518244743347,
                       'source': '14219_3_EN_-_Bearing_failures_LOW_pdf_preview_medium.pdf',
                       'source_id': 2},
                      {'page': 13,
                       'score': 0.5754793882369995,
                       'source': '14219_3_EN_-_Bearing_failures_LOW_pdf_preview_medium.pdf',
                       'source_id': 3}],
 'rag_generated_advice': '1. Likely issue: Excessive load or excessive speed '
                         'leading to increased operating temperature and '
                         'potential out-of-round housing.\n'
                         '2. Immediate action: Reduce the load or

---
## 10. Demo Queries

In [ ]:
# Demo 1: pure sensor diagnosis
ex = test_records[1]
print('Sensor input:')
print(ex['input'])
print('\nExpected:', normalize_label(ex['output']))
print('Predicted:', predict_fault_label(ex['instruction'], ex['input']))

Sensor input:
Max=0.7965, Min=-0.6905, Mean=0.0313, StdDev=0.1835, RMS=0.1861, Skewness=0.0199, Kurtosis=0.8290, CrestFactor=4.2791, FormFactor=5.9470

Expected: Inner Race - Medium
Predicted: Inner Race - Medium


In [ ]:
# Demo 2: pure manual lookup
manual_query = 'bearing inner race fault repair and lubrication procedure'
for ctx in retrieve_manual_context(manual_query, top_k=3):
    print(f"{ctx['source']} page {ctx['page']} score={ctx['score']:.3f}")
    print(ctx['text'][:700], '\n')

14219_3_EN_-_Bearing_failures_LOW_pdf_preview_medium.pdf page 23 score=0.619
was used . – The seal was installed correctly . – There is no seal wear, seal damage or lubricant leakage . • The relubrication interval may need to be shortened . Supplying smaller quantities of fresh grease more frequently can help purge contaminated grease from the bearing/housing cavity . • Consider replacing open bearings with sealed bearings . 28 Solids from manufacturing or previous bearing failures in the housing Considerations during cleaning or assembly and about lubricant cleanliness: • Denting of the bearing contact surfaces can occur when solid contaminants are left in the bearing housing from a previous failure, from wear of other components such as gears, or from contaminated  

14219_3_EN_-_Bearing_failures_LOW_pdf_preview_medium.pdf page 98 score=0.607
98 Appendix D: Collecting information 8 Appendices Whenever bearing damage or a failure occurs, it is very important to collect and document an

In [ ]:
# Demo 3: full FaultSense mixed query with RAG-generated repair advice
ex = test_records[2]
result = faultsense_diagnose(ex['instruction'], ex['input'], top_k=4)
recommendation = result['recommendation']

print('Sensor input:')
print(ex['input'])
print('\nExpected label:', normalize_label(ex['output']))
print('Predicted label:', result['fault_label'])
print('RAG query:', result['rag_query'])

print('\nDiagnosis:')
print(recommendation['diagnosis'])

print('\nRAG-generated repair advice:')
print(recommendation['rag_generated_advice'])

print('\nManual citations used for grounding:')
for citation in recommendation['manual_citations']:
    print(
        f"[Source {citation['source_id']}] {citation['source']} "
        f"page {citation['page']} score={citation['score']:.3f}"
    )

print('\nRetrieved context previews:')
for i, ctx in enumerate(result['retrieved_context'], start=1):
    preview = ctx['text'][:450].replace('\n', ' ')
    print(f"\n[Source {i}] {ctx['source']} page {ctx['page']} score={ctx['score']:.3f}")
    print(preview, '...')


Sensor input:
Max=0.4218, Min=-0.4540, Mean=0.0141, StdDev=0.1407, RMS=0.1413, Skewness=-0.0626, Kurtosis=-0.0831, CrestFactor=2.9848, FormFactor=10.0355

Expected label: Outer Race - Medium
Predicted label: Outer Race - Medium
RAG query: Outer Race bearing fault Medium severity bearing outer race defect vibration housing alignment repair replacement maintenance repair inspect soon planned replacement

Diagnosis:
Outer Race bearing fault at Medium severity

RAG-generated repair advice:
1. Likely issue: Uneven support of bearing rings leading to outer race wear.
2. Immediate action: Reduce load and speed if possible.
3. Inspection or repair steps:
   - Verify the flatness and rigidity of the housing base.
   - Check the roundness of the shaft and housing seats.
   - Ensure proper fit between the bearing and housing.
4. Operating priority: Medium priority; schedule maintenance soon.
5. Citations by source number: [Source 1], [Source 3]

Explanation: The context indicates that uneven supp

---
## 11. Optional: Save Demo Outputs

In [ ]:
OUTPUT_PATH = PROJECT_DIR / 'evaluation' / 'faultsense_rag_demo_outputs.json'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

SAVE_LIMIT = 5  # Increase to 10+ for more report examples; each item runs generation.

demo_outputs = []
for i, ex in enumerate(test_records[:SAVE_LIMIT], start=1):
    print(f'Running full FaultSense pipeline for example {i}/{SAVE_LIMIT}...')
    out = faultsense_diagnose(ex['instruction'], ex['input'], top_k=3)
    rec = out['recommendation']
    demo_outputs.append({
        'index': i - 1,
        'sensor_input': ex['input'],
        'expected_label': normalize_label(ex['output']),
        'predicted_label': out['fault_label'],
        'rag_query': out['rag_query'],
        'diagnosis': rec['diagnosis'],
        'rag_generated_advice': rec['rag_generated_advice'],
        'manual_citations': rec['manual_citations'],
        'retrieved_context_previews': [
            {
                'source_id': j + 1,
                'source': c['source'],
                'page': c['page'],
                'score': c['score'],
                'preview': c['text'][:600],
            }
            for j, c in enumerate(out['retrieved_context'])
        ],
    })

OUTPUT_PATH.write_text(json.dumps(demo_outputs, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', OUTPUT_PATH)


Running full FaultSense pipeline for example 1/5...
Running full FaultSense pipeline for example 2/5...
Running full FaultSense pipeline for example 3/5...
Running full FaultSense pipeline for example 4/5...
Running full FaultSense pipeline for example 5/5...
Saved: /content/drive/MyDrive/FaultSense_Project/evaluation/faultsense_rag_demo_outputs.json
